# Notebook 00c. Partición por días (prueba de fuga)
Prueba pedida por el ingeniero: el **test está formado por días completos de muestreo que train nunca ve**. Si las métricas se desploman respecto a la partición por grupos, había fuga de datos.

**Qué hace:**
1. Lee el día de captura del nombre del archivo (`IMG_20250426_083809` → 2025-04-26).
2. **Test = 4 días completos:** 2025-01-08, 2025-01-14, 2025-02-09 y 2025-02-15 (≈10 % de cada clase). No se eligió el 2025-05-10 porque las fotos con nombre de 13 dígitos se exportaron esa misma tarde y podrían ser de ese día. Tampoco los días contiguos a 2025-06-22, el día más grande.
3. Las fotos sin hora (13 dígitos, WhatsApp) van a train. Los 4 enlaces visuales entre fotos de días distintos se revisaron a ojo y eran hojas diferentes, así que no se usan. Si alguna está unida a una foto de test por similitud visual (`grupos_enlaces.csv` del caché v2), pasa a test junto con ella.
4. **Val (≈10 %)** sale de los días de train, por grupos: fotos de la misma clase con ≤ 60 s entre sí + enlaces visuales. Se reparten grupos completos cuidando la proporción de clases (semilla 42).
5. Estadísticas y pesos de clase solo con train.

Settings: Accelerator **None**, Internet **Off**. Input: dataset **banana-cache-224-v2** (X224.npy, meta.csv, grupos_enlaces.csv). Salida: caché v3 (X224.npy igual, meta.csv y stats.json nuevos).

In [ ]:
# CELDA 1: cargar caché v2 y leer el día de captura
import os, re, glob, json, shutil
from datetime import datetime
import numpy as np, pandas as pd

CACHE = glob.glob("/kaggle/input/**/X224.npy", recursive=True)
assert len(CACHE) == 1, CACHE
DIR, OUT = os.path.dirname(CACHE[0]), "/kaggle/working"
X = np.load(CACHE[0], mmap_mode="r")
meta = pd.read_csv(f"{DIR}/meta.csv")[["file_name", "class_name", "label", "sample_id"]]
enl = pd.read_csv(f"{DIR}/grupos_enlaces.csv")
assert X.shape == (len(meta), 224, 224, 3) and len(meta) == 2354
CLASES = ["sigatoka", "cordana", "sanas"]
TEST_DAYS = ["2025-01-08", "2025-01-14", "2025-02-09", "2025-02-15"]
T_VAL = 60

def hora(f):
    s = re.sub(r"\(\d+\)$", "", f.rsplit(".", 1)[0])
    r = re.fullmatch(r"(?:IMG_)?(\d{8})_(\d{6})", s)
    return datetime.strptime(r.group(1) + r.group(2), "%Y%m%d%H%M%S") if r else None
h = [hora(f) for f in meta.file_name]
meta["t"] = [x.timestamp() if x is not None else np.nan for x in h]
meta["day"] = [x.strftime("%Y-%m-%d") if x is not None else "sin_hora" for x in h]
pos = {s: i for i, s in enumerate(meta.sample_id)}
vis = [(pos[a], pos[b]) for a, b, k in zip(enl.sample_a, enl.sample_b, enl.motivo) if k != "rafaga_tiempo"]
# 4 enlaces visuales unen fotos de dias distintos; revisados a ojo, son hojas diferentes (falsos positivos), se descartan
dia = meta.day.values
vis = [(a, b) for a, b in vis if dia[a] == dia[b] or "sin_hora" in (dia[a], dia[b])]
print("enlaces visuales usados:", len(vis))
print(pd.crosstab(meta.day, meta.class_name)[CLASES])

In [ ]:
# CELDA 2: test = dias completos (+ fotos unidas visualmente a ellas)
meta["split"] = "train"
meta.loc[meta.day.isin(TEST_DAYS), "split"] = "test"
cambio = True
while cambio:
    cambio = False
    for a, b in vis:
        sa, sb = meta.split.iat[a], meta.split.iat[b]
        if (sa == "test") != (sb == "test"):
            otro = b if sa == "test" else a
            assert meta.day.iat[otro] == "sin_hora" or meta.day.iat[otro] in TEST_DAYS, ("enlace visual con otro dia", meta.sample_id.iat[a], meta.sample_id.iat[b])
            meta.loc[otro, "split"] = "test"; cambio = True
print("test:", (meta.split == "test").sum(), "| fotos sin hora que pasaron a test:", ((meta.split == "test") & (meta.day == "sin_hora")).sum())

In [ ]:
# CELDA 3: val por grupos (<= 60 s misma clase + enlaces visuales) dentro de los dias de train
par = np.arange(len(meta))
def raiz(x):
    while par[x] != x:
        par[x] = par[par[x]]; x = par[x]
    return x
def unir(a, b):
    ra, rb = raiz(a), raiz(b)
    if ra != rb: par[max(ra, rb)] = min(ra, rb)
c = meta[meta.t.notna()].sort_values(["class_name", "t"])
for _, g in c.groupby("class_name"):
    ii, tt = g.index.values, g.t.values
    for k in range(1, len(ii)):
        if tt[k] - tt[k - 1] <= T_VAL: unir(ii[k - 1], ii[k])
for a, b in vis: unir(a, b)
meta["group_id"] = [raiz(i) for i in range(len(meta))]
G = meta[meta.split == "train"].groupby("group_id").agg(c=("class_name", "first"), n=("class_name", "size"), nc=("class_name", "nunique"))
assert G.nc.max() == 1
G["r"] = np.random.default_rng(42).random(len(G)); G = G.sort_values("r")
meta_tot = meta.class_name.value_counts()
obj = {k: 0.10 * meta_tot[k] for k in CLASES}; cur = {k: 0 for k in CLASES}
for gid, r in G.iterrows():
    if cur[r.c] + r.n <= obj[r.c] + 0.5:
        meta.loc[meta.group_id == gid, "split"] = "val"; cur[r.c] += r.n
assert meta.groupby("group_id").split.nunique().max() == 1
print(pd.crosstab(meta.class_name, meta.split, margins=True)[["train", "val", "test", "All"]])
print((pd.crosstab(meta.class_name, meta.split, normalize="columns") * 100).round(1))

In [ ]:
# CELDA 4: estadisticas SOLO con train y guardar el caché v3
tr_idx = np.where(meta.split == "train")[0]
s, ss, n = np.zeros(3), np.zeros(3), 0
for i in tr_idx:
    x = X[i].astype(np.float64) / 255.0
    s += x.sum((0, 1)); ss += (x ** 2).sum((0, 1)); n += x.shape[0] * x.shape[1]
mean = s / n; std = np.sqrt(ss / n - mean ** 2)
n_c = np.bincount(meta.label.values[tr_idx], minlength=3)
stats = {"mean_train": mean.tolist(), "std_train": std.tolist(), "imagenes_train_por_clase": n_c.tolist(),
         "pesos_clase_train": (len(tr_idx) / (3 * n_c)).tolist(), "clases": CLASES,
         "split": "test por dias completos (" + ", ".join(TEST_DAYS) + "); val por grupos de 60 s; semilla 42"}
json.dump(stats, open(f"{OUT}/stats.json", "w"), indent=1)
meta[["file_name", "class_name", "label", "sample_id", "split", "group_id", "day"]].to_csv(f"{OUT}/meta.csv", index=False)
if os.path.abspath(CACHE[0]) != os.path.abspath(f"{OUT}/X224.npy"):
    shutil.copy(CACHE[0], f"{OUT}/X224.npy")
print(json.dumps(stats, indent=1))

In [ ]:
# CELDA 5: verificaciones
m = pd.read_csv(f"{OUT}/meta.csv")
te, tr, va = (m.split == "test").values, (m.split == "train").values, (m.split == "val").values
assert not set(m.day[te]) & set(m.day[tr]) - {"sin_hora"}, "un dia de test aparece en train"
assert not set(m.day[te]) & set(m.day[va]) - {"sin_hora"}, "un dia de test aparece en val"
sp = m.split.values
assert all(sp[a] == sp[b] for a, b in vis), "un enlace visual cruza particiones"
print("dias de test:", sorted(set(m.day[te]) - {"sin_hora"}), "| dias de test que aparecen en train: 0 | enlaces visuales que cruzan: 0")
print("CACHE V3 OK | train/val/test:", m.split.value_counts().to_dict())